In [43]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)

sales_raw = pd.DataFrame({
    "date": [
        "2026-01-01",
        "2026-01-02",
        "2026-01-03",
        "bad-date",
        "2026-01-05",
        "2026-01-06",
        "2026-01-07"
    ],
    "sales": [120, 145, 138, 160, 172, 168, 190]
})

dates_90 = pd.date_range("2026-02-01", periods=90, freq="D")

store = pd.DataFrame({
    "date": dates_90,
    "orders": rng.integers(40, 120, size=90),
    "revenue": np.round(rng.normal(7000, 1200, size=90), 2),
    "returns": rng.integers(0, 10, size=90)
})

weekend = store["date"].dt.dayofweek >= 5
store.loc[weekend, "orders"] += 20
store.loc[weekend, "revenue"] += 1500

sensor_dates = pd.date_range("2026-05-01", periods=35, freq="D")
drop_idx = [3, 4, 11, 18, 19, 20, 28]
sensor_dates = sensor_dates.delete(drop_idx)

sensor = pd.DataFrame({
    "timestamp": sensor_dates,
    "temperature": np.round(rng.normal(20, 3, size=len(sensor_dates)), 1)
})

traffic_times = pd.date_range("2026-06-01", periods=24 * 28, freq="h")
hours = traffic_times.hour
daily_cycle = 100 + 60 * np.sin((hours - 8) / 24 * 2 * np.pi)

traffic = pd.DataFrame({
    "timestamp": traffic_times,
    "visits": np.maximum(
        5,
        np.round(daily_cycle + rng.normal(0, 15, len(traffic_times)))
    ).astype(int)
})
traffic.loc[traffic["timestamp"].dt.dayofweek >= 5, "visits"] += 25


In [44]:
# HW 1
# 1. Inspect the data types in `sales_raw`
sales_raw.dtypes
# 3. Report invalid dates as missing datetime values ("NaT")
# 4. Store the result in a new column `date_parsed`
sales_raw['date_parsed'] = pd.to_datetime(sales_raw['date'], errors='coerce')
sales_raw.head()
# 5. Count dates are missing
sales_raw['date_parsed'].isna().sum()


np.int64(1)

In [45]:
# HW 2
# Add new columns to `sales_raw`: year, month, day, weekday
sales_raw['year'] = sales_raw['date_parsed'].dt.year
sales_raw['month'] = sales_raw['date_parsed'].dt.month
sales_raw['day'] = sales_raw['date_parsed'].dt.day
sales_raw['weekday'] = sales_raw['date_parsed'].dt.day_name()
sales_raw.head()

,date,sales,date_parsed,year,month,day,weekday
0,2026-01-01,120,2026-01-01,2026.0,1.0,1.0,Thursday
1,2026-01-02,145,2026-01-02,2026.0,1.0,2.0,Friday
2,2026-01-03,138,2026-01-03,2026.0,1.0,3.0,Saturday
3,bad-date,160,NaT,NaN,NaN,NaN,NaN
4,2026-01-05,172,2026-01-05,2026.0,1.0,5.0,Monday


In [46]:
# HW 3
# 1. Make a copy of `store` called `store_ts`
store_ts = store.copy()
# 2. Set `date` as the index and sort it
store_ts = store_ts.set_index('date').sort_index()
# 3. Select only the rows between 2026-02-10 and 2026-02-20
mask = store_ts.loc['2026-02-10':'2026-02-20']
mask.head()
# 4. Find the total revenue during that period
total_rev = mask['revenue'].sum()
total_rev
store_ts


,orders,revenue,returns
date,,,
2026-02-01,67,8778.59,2
2026-02-02,101,7140.02,0
2026-02-03,92,7262.43,1
2026-02-04,75,8045.71,9
2026-02-05,74,7268.31,4
...,...,...,...
2026-04-27,96,6232.30,9
2026-04-28,47,5888.11,1
2026-04-29,64,6532.23,8


In [47]:
# HW 4
# 1. For `store_ts`, calculate weekly:
weekNumber = store_ts.index.isocalendar().week
store_ts['week'] = weekNumber
store_ts.head()
#    - total orders
store_ts.groupby('week')['orders'].sum()
#    - total revenue
store_ts.groupby('week')['revenue'].sum()
#    - total returns
store_ts.groupby('week')['returns'].sum()
# 2. Calculate monthly total revenue
monthNumber = store_ts.index.month
store_ts['month'] = monthNumber
monthly_revenue = store_ts.groupby('month')['revenue'].sum()
# 3. Store the results in:
#    - `weekly_store`
weekly_store = store_ts.groupby('week')[['orders', 'revenue', 'returns']].sum()
#    - `monthly_revenue`
monthly_revenue = store_ts.groupby('month')['revenue'].sum()
weekly_store
# Answer:
# 1. Which week had the highest total revenue?
weekly_store['revenue'].idxmax()
# 2. Which month had the highest total revenue?
monthly_revenue.idxmax()


np.int32(3)

In [48]:
# HW 5
# With `store_ts`, create a weekly (resampled) summary containing:
# - `orders`: `sum` and `mean`
# - `revenue`: `sum`, `mean`, and `max`
# - `returns`: `sum`
weekly_summary = store_ts.resample('W').agg({
    'orders': ['sum', 'mean'],
    'revenue': ['sum', 'mean', 'max'],
    'returns': 'sum'
})
weekly_summary
# Save the result as `weekly_summary`


orders              revenue                        returns
              sum       mean       sum         mean       max     sum
date                                                                 
2026-02-01     67  67.000000   8778.59  8778.590000   8778.59       2
2026-02-08    631  90.142857  54959.20  7851.314286   8846.94      29
2026-02-15    638  91.142857  50758.01  7251.144286  10293.93      20
2026-02-22    605  86.428571  51453.87  7350.552857   9353.47      34
2026-03-01    661  94.428571  49887.97  7126.852857   8029.57      36
2026-03-08    536  76.571429  52921.00  7560.142857   9250.71      25
2026-03-15    655  93.571429  49638.21  7091.172857   9084.37      25
2026-03-22    652  93.142857  52736.77  7533.824286   8381.82      40
2026-03-29    546  78.000000  44752.23  6393.175714   8979.73      32
2026-04-05    683  97.571429  51055.87  7293.695714   8559.07      25
2026-04-12    585  83.571429  48887.09  6983.870000   8008.37      41
2026-04-19    518  74.000000  52163.30  7451.900000   8922.13      42
2026-04-26    655  93.571429  55869.98  7981.425714  10255.96      22
2026-05-03    414  82.800000  31762.80  6352.560000   7762.18      25

In [51]:
# HW 6
# Based on `store_ts['revenue']`, calculate:
# - `previous_revenue` with `shift(1)`
store_ts['previous_revenue'] = store_ts['revenue'].shift(1)
# - `revenue_change` with `diff()`
store_ts['revenue_change'] = store_ts['revenue'].diff()
# - `revenue_pct_change` iwth `pct_change()`
store_ts['revenue_pct_change'] = store_ts['revenue'].pct_change()
# - `revenue_vs_last_week` with `pct_change(7)`
store_ts['revenue_vs_last_week'] = store_ts['revenue'].pct_change(7)
store_ts.head(10)


,orders,revenue,returns,week,month,previous_revenue,revenue_change,revenue_pct_change,revenue_vs_last_week
date,,,,,,,,,
2026-02-01,67,8778.59,2,5,2,NaN,NaN,NaN,NaN
2026-02-02,101,7140.02,0,6,2,8778.59,-1638.57,-0.186655,NaN
2026-02-03,92,7262.43,1,6,2,7140.02,122.41,0.017144,NaN
2026-02-04,75,8045.71,9,6,2,7262.43,783.28,0.107854,NaN
2026-02-05,74,7268.31,4,6,2,8045.71,-777.40,-0.096623,NaN
2026-02-06,108,7814.70,4,6,2,7268.31,546.39,0.075174,NaN
2026-02-07,66,8581.09,4,6,2,7814.70,766.39,0.098070,NaN
2026-02-08,115,8846.94,7,6,2,8581.09,265.85,0.030981,0.007786
2026-02-09,56,7757.55,0,7,2,8846.94,-1089.39,-0.123137,0.086489


In [52]:
# HW 7
# 1. Resample weekly total revenue from `store_ts` to daily freq
# 2. Forward-fill (ffill) the most recent weekly value
# 3. Save the result as `daily_from_weekly`
daily_from_weekly = store_ts['revenue'].resample('W').sum().resample('D').ffill()
daily_from_weekly


date
2026-02-01     8778.59
2026-02-02     8778.59
2026-02-03     8778.59
2026-02-04     8778.59
2026-02-05     8778.59
                ...   
2026-04-29    55869.98
2026-04-30    55869.98
2026-05-01    55869.98
2026-05-02    55869.98
2026-05-03    31762.80
Freq: D, Name: revenue, Length: 92, dtype: float64